# Robust tuning (flat-top cavity)

> **Uncertainty quantification plugs into every analysis.** In cavsim2d, a run is deterministic — one geometry, one answer — until you add a `uq_config`. With it, the analysis is evaluated over a small Stroud-3 quadrature of the uncertain geometry and returns a **distribution** (mean and standard deviation) instead of a single number. The *same* pattern works for eigenmode, tuning (robust tuning), wakefield, and optimisation (robust optimisation) — only multipacting does not yet take a `uq_config`.

These advanced examples deliberately span several cavity types (spline, RF gun, flat-top, elliptical, pillbox) — a good stress test of the model-agnostic machinery.

**Robust tuning** is ordinary tuning with a `uq_config`: each tuning evaluation is averaged over the quadrature of the uncertain geometry, so the reached figures of merit carry the spread the tolerances imply. We tune an **elliptical flat-top** cell to 790 MHz.

In [1]:
import os
import tempfile
import json

import pandas as pd

from cavsim2d import Study, EllipticalCavityFlatTop
from cavsim2d.utils.style import apply_style

apply_style()

## 1. Deterministic tuning

Parameters `[A, B, a, b, Ri, L, Req, l]` in mm (the trailing `l` is the flat-top plateau length). Tune the equator radius `Req` to hit 790 MHz.

In [2]:
ft = [62.22, 66.13, 30.22, 23.11, 80, 93.5, 171.20, 20]
cavs = Study(os.path.join(tempfile.mkdtemp(), 'ft_tune'))
cavs.add_cavity([EllipticalCavityFlatTop(1, ft, ft, ft, beampipe='none')], ['FT'])

cavs.run_tune({
    'freqs': 790.0,
    'cell_type': {'mid-cell': 'Req'},
    'processes': 1,
    'eigenmode_config': {'boundary_conditions': 'mm', 'mesh_config': {'h': 8, 'p': 2}},
})
res = cavs.cavities_list[0].tune.qois['mid-cell']
print(f"tuned Req = {res['parameters']['Req_m']:.3f} mm at {res['FREQ']:.3f} MHz")

tuned Req = 172.376 mm at 790.000 MHz


## 2. Robust tuning

Add a `uq_config` with a 2% spread on the semi-axes `A, B`. Same call, now UQ-averaged.

In [3]:
cavs2 = Study(os.path.join(tempfile.mkdtemp(), 'ft_rtune'))
cav = EllipticalCavityFlatTop(1, ft, ft, ft, beampipe='none')
cavs2.add_cavity([cav], ['FT'])

cavs2.run_tune({
    'freqs': 790.0,
    'cell_type': {'mid-cell': 'Req'},
    'processes': 1,
    'eigenmode_config': {'boundary_conditions': 'mm', 'mesh_config': {'h': 8, 'p': 2}},
    'uq_config': {
        'variables': ['A', 'B'],
        'objectives': ['monopole:freq [MHz]', 'monopole:R/Q [Ohm]'],
        'method': ['stroud3'], 'delta': [0.02, 0.02], 'processes': 1,
        'cell_type': 'mid-cell', 'cell_complexity': 'simplecell',
    },
})

## 3. Spread of the tuned figures of merit

Read from `<cav>/tuned/tune_info/uq_tune_results.json`. As in any robust tuning, the **frequency is pinned** (tuned onto target for every node, so ~zero spread) and the residual uncertainty surfaces in R/Q.

In [4]:
uq = json.load(open(os.path.join(cav.tune.folder, 'uq_tune_results.json')))
stats = next(iter(uq.values()))
pd.DataFrame({q.split(':', 1)[-1]: {'mean': v['expe'][0], 'std': v['stdDev'][0]}
             for q, v in stats.items()}).T.round(6)

,mean,std
freq [MHz],789.999997,0.0


See also: [eigenmode UQ](eigenmode_uq.ipynb) and [robust optimisation](robust_optimisation.ipynb).